# Rx PD Model

Explores the Lending Default training and holdout data, then runs the **restaurant-level modeling pipeline** from `Restaurant_Default_Model.ipynb`: feature engineering, stratified train/test split, WoE binning, `ShapRFECV`, tuned logistic regression, and diagnostics (ROC/KS, deciles, VIF, correlation, SHAP).


In [1]:
# Optional: `toad` scorecard helpers
#
# If you have a diagnostic cell that does `site.addsitedir(...)` then `import toad`, you can
# **delete that cell**. On some macOS/Python setups the `toad` wheel installs metadata but
# `importlib.util.find_spec("toad")` stays `None`, so `import toad` still raises ModuleNotFoundError.
# This notebook pipeline does **not** require `toad`.
pass


In [2]:
import pandas as pd
import numpy as np
import random

import seaborn as sns #For plotting graphs
sns.set_style("whitegrid")

import matplotlib.pyplot as plt #For plotting graphs
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100 # 200 e.g. is really fine, but slower

import seaborn as sns #For plotting graphs
sns.set_style("whitegrid")

from sklearn.model_selection import train_test_split

from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV

from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, \
accuracy_score, classification_report, confusion_matrix, mean_squared_error, \
balanced_accuracy_score,roc_curve,auc

# Classifier Libraries
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

# Recursive Feature Elimination
# from probatus.feature_elimination import ShapRFECV

# import shap
# shap.initjs()

# import toad  # optional dependency; disabled

# from optbinning import OptimalBinning


import os
from pathlib import Path

# Resolve data file location robustly regardless of execution cwd
CANDIDATE_DIRS = [
    Path.cwd(),
    Path.cwd() / 'data',
    Path('/Users/pradark/Documents/011. Work/Toast/Principal Data Scientist Capital Case Study'),
    Path('/Users/pradark/Documents/011. Work/Toast/Principal Data Scientist Capital Case Study/data'),
]


def _resolve_data_file(filename: str) -> str:
    for d in CANDIDATE_DIRS:
        fp = d / filename
        if fp.exists():
            return str(fp)
    raise FileNotFoundError(f'Could not locate {filename} in candidate dirs: {CANDIDATE_DIRS}')

import warnings
warnings.filterwarnings("ignore")

## Training Dataset

In [3]:
# Load training data
train_account = pd.read_csv(_resolve_data_file('Lending_default_train_account.csv'))
train_tx = pd.read_csv(_resolve_data_file('Lending_default_train_tx.csv'))
train_label = pd.read_csv(_resolve_data_file('Lending_default_train_label.csv'))

print('=' * 80)
print('TRAINING ACCOUNT DATA')
print('=' * 80)
print(f'Shape: {train_account.shape}')
print(f'\nColumns: {list(train_account.columns)}')
print(f'\nData Types:\n{train_account.dtypes}')
print(f'\nMissing Values:\n{train_account.isnull().sum()}')
print(f'\nFirst 5 rows:')
print(train_account.head())
print(f'\nBasic Statistics:')
print(train_account.describe())

TRAINING ACCOUNT DATA
Shape: (10812, 5)

Columns: ['Unnamed: 0', 'Restaurant_ID', 'Ownership_type', 'Restaurant_catagory', 'Market_segment']

Data Types:
Unnamed: 0              int64
Restaurant_ID          object
Ownership_type         object
Restaurant_catagory    object
Market_segment         object
dtype: object

Missing Values:
Unnamed: 0               0
Restaurant_ID            0
Ownership_type         220
Restaurant_catagory      3
Market_segment           0
dtype: int64

First 5 rows:
   Unnamed: 0                         Restaurant_ID Ownership_type  \
0           0  cc3c8fb4-84c6-49c8-bcc7-6d23c626dad5    Partnership   
1           1  2c35efdf-410f-4188-bfe9-9777a737a107    Partnership   
2           2  6fa519c2-e88c-4012-a981-4e4fa55be951    Corporation   
3           3  541dd2c4-f612-4d28-9fae-7e28c9ee8584            LLC   
4           4  f010f8ec-7440-4676-b7b3-ed3f4e27baf4    Corporation   

  Restaurant_catagory Market_segment  
0   QSR - Fast Casual            SMB  
1  

In [4]:
print('=' * 80)
print('TRAINING TRANSACTION DATA')
print('=' * 80)
print(f'Shape: {train_tx.shape}')
print(f'\nColumns: {list(train_tx.columns)}')
print(f'\nData Types:\n{train_tx.dtypes}')
print(f'\nMissing Values:\n{train_tx.isnull().sum()}')
print(f'\nFirst 5 rows:')
print(train_tx.head())
print(f'\nBasic Statistics:')
print(train_tx.describe())

TRAINING TRANSACTION DATA
Shape: (3510679, 5)

Columns: ['Unnamed: 0', 'Restaurant_ID', 'Tx_date', 'processing_volume', 'Tx_hours']

Data Types:
Unnamed: 0             int64
Restaurant_ID         object
Tx_date               object
processing_volume    float64
Tx_hours             float64
dtype: object

Missing Values:
Unnamed: 0           0
Restaurant_ID        0
Tx_date              0
processing_volume    0
Tx_hours             0
dtype: int64

First 5 rows:
   Unnamed: 0                         Restaurant_ID     Tx_date  \
0           0  cc3c8fb4-84c6-49c8-bcc7-6d23c626dad5  2021-02-02   
1           1  cc3c8fb4-84c6-49c8-bcc7-6d23c626dad5  2021-02-03   
2           2  cc3c8fb4-84c6-49c8-bcc7-6d23c626dad5  2021-02-04   
3           3  cc3c8fb4-84c6-49c8-bcc7-6d23c626dad5  2021-02-05   
4           4  cc3c8fb4-84c6-49c8-bcc7-6d23c626dad5  2021-02-06   

   processing_volume  Tx_hours  
0            3778.49      12.0  
1            4456.12      14.0  
2            4868.75      12.0  
3

         Unnamed: 0  processing_volume      Tx_hours
count  3.510679e+06       3.510679e+06  3.510679e+06
mean   1.755339e+06       3.573939e+03  8.851561e+00
std    1.013446e+06       5.342567e+03  4.575507e+00
min    0.000000e+00       0.000000e+00  0.000000e+00
25%    8.776695e+05       6.288150e+02  6.000000e+00
50%    1.755339e+06       1.942680e+03  1.000000e+01
75%    2.633008e+06       4.445410e+03  1.200000e+01
max    3.510678e+06       6.077315e+05  2.400000e+01


In [5]:
print('=' * 80)
print('TRAINING LABEL DATA')
print('=' * 80)
print(f'Shape: {train_label.shape}')
print(f'\nColumns: {list(train_label.columns)}')
print(f'\nData Types:\n{train_label.dtypes}')
print(f'\nMissing Values:\n{train_label.isnull().sum()}')
print(f'\nFirst 5 rows:')
print(train_label.head())
print(f'\nLabel Distribution:')
print(train_label.iloc[:, -1].value_counts())

TRAINING LABEL DATA
Shape: (10812, 3)

Columns: ['Unnamed: 0', 'Restaurant_ID', 'loan_default']

Data Types:
Unnamed: 0         int64
Restaurant_ID     object
loan_default     float64
dtype: object

Missing Values:
Unnamed: 0       0
Restaurant_ID    0
loan_default     0
dtype: int64

First 5 rows:
   Unnamed: 0                         Restaurant_ID  loan_default
0           0  cc3c8fb4-84c6-49c8-bcc7-6d23c626dad5           0.0
1           1  2c35efdf-410f-4188-bfe9-9777a737a107           0.0
2           2  6fa519c2-e88c-4012-a981-4e4fa55be951           1.0
3           3  541dd2c4-f612-4d28-9fae-7e28c9ee8584           0.0
4           4  f010f8ec-7440-4676-b7b3-ed3f4e27baf4           0.0

Label Distribution:
loan_default
0.0    9787
1.0    1025
Name: count, dtype: int64


## Holdout Dataset

In [6]:
# Load holdout data
holdout_account = pd.read_csv(_resolve_data_file('Lending_default_holdout_account.csv'))
holdout_tx = pd.read_csv(_resolve_data_file('Lending_default_holdout_tx.csv'))

print('=' * 80)
print('HOLDOUT ACCOUNT DATA')
print('=' * 80)
print(f'Shape: {holdout_account.shape}')
print(f'\nColumns: {list(holdout_account.columns)}')
print(f'\nData Types:\n{holdout_account.dtypes}')
print(f'\nMissing Values:\n{holdout_account.isnull().sum()}')
print(f'\nFirst 5 rows:')
print(holdout_account.head())
print(f'\nBasic Statistics:')
print(holdout_account.describe())

HOLDOUT ACCOUNT DATA
Shape: (4514, 5)

Columns: ['Unnamed: 0', 'Restaurant_ID', 'Ownership_type', 'Restaurant_catagory', 'Market_segment']

Data Types:
Unnamed: 0              int64
Restaurant_ID          object
Ownership_type         object
Restaurant_catagory    object
Market_segment         object
dtype: object

Missing Values:
Unnamed: 0              0
Restaurant_ID           0
Ownership_type         72
Restaurant_catagory     3
Market_segment          0
dtype: int64

First 5 rows:
   Unnamed: 0                         Restaurant_ID       Ownership_type  \
0           0  7ae540d4-fd25-488e-b081-800847db321e  Sole Proprietorship   
1           1  44c79166-f2af-47f8-9b39-6708174a6328                  LLC   
2           2  851df45d-4ef0-482d-b13f-84ce70cd8ab9  Sole Proprietorship   
3           3  ab873be0-28c6-460d-a84a-8381a34eba53                  LLC   
4           4  c4f4aea8-fff8-439b-8a1a-f9a6d8d4a4ad  Sole Proprietorship   

   Restaurant_catagory Market_segment  
0  FSR - Cas

In [7]:
print('=' * 80)
print('HOLDOUT TRANSACTION DATA')
print('=' * 80)
print(f'Shape: {holdout_tx.shape}')
print(f'\nColumns: {list(holdout_tx.columns)}')
print(f'\nData Types:\n{holdout_tx.dtypes}')
print(f'\nMissing Values:\n{holdout_tx.isnull().sum()}')
print(f'\nFirst 5 rows:')
print(holdout_tx.head())
print(f'\nBasic Statistics:')
print(holdout_tx.describe())

HOLDOUT TRANSACTION DATA
Shape: (1471016, 5)

Columns: ['Unnamed: 0', 'Restaurant_ID', 'Tx_date', 'processing_volume', 'Tx_hours']

Data Types:
Unnamed: 0             int64
Restaurant_ID         object
Tx_date               object
processing_volume    float64
Tx_hours             float64
dtype: object

Missing Values:
Unnamed: 0           0
Restaurant_ID        0
Tx_date              0
processing_volume    0
Tx_hours             0
dtype: int64

First 5 rows:
   Unnamed: 0                         Restaurant_ID     Tx_date  \
0           0  7ae540d4-fd25-488e-b081-800847db321e  2021-07-02   
1           1  7ae540d4-fd25-488e-b081-800847db321e  2021-07-03   
2           2  7ae540d4-fd25-488e-b081-800847db321e  2021-07-04   
3           3  7ae540d4-fd25-488e-b081-800847db321e  2021-07-05   
4           4  7ae540d4-fd25-488e-b081-800847db321e  2021-07-06   

   processing_volume  Tx_hours  
0            3223.37      16.0  
1            2555.51      16.0  
2            2774.95      14.0  
3 

## Dataset Summary

In [8]:
print('=' * 80)
print('DATASET SUMMARY')
print('=' * 80)
print(f'\nTraining Set:')
print(f'  - Accounts: {len(train_account)}')
print(f'  - Transactions: {len(train_tx)}')
print(f'  - Labels: {len(train_label)}')

print(f'\nHoldout Set:')
print(f'  - Accounts: {len(holdout_account)}')
print(f'  - Transactions: {len(holdout_tx)}')

print(f'\nTotal:')
print(f'  - Total Accounts: {len(train_account) + len(holdout_account)}')
print(f'  - Total Transactions: {len(train_tx) + len(holdout_tx)}')

DATASET SUMMARY

Training Set:
  - Accounts: 10812
  - Transactions: 3510679
  - Labels: 10812

Holdout Set:
  - Accounts: 4514
  - Transactions: 1471016

Total:
  - Total Accounts: 15326
  - Total Transactions: 4981695


## Modeling pipeline (from `Restaurant_Default_Model.ipynb`)

**Restaurant-level aggregates**, **WoE** (`optbinning`), **RFE** (`probatus.ShapRFECV`),
**tuned logistic regression**, and **diagnostics** (ROC / KS, decile actual vs predicted, metrics, VIF, correlation, SHAP).

Install if needed: `optbinning`, `probatus`, `shap`, `statsmodels`, `scikit-learn`, `matplotlib`, `seaborn`.


In [9]:
# Build rolling time-series features by Restaurant_ID and snapshot Tx_date
LABEL_KEY = 'Restaurant_ID'

# Standardize dataframe names with df_ prefix
df_train_account = train_account.copy()
df_train_tx = train_tx.copy()
df_train_label = train_label.copy()
df_holdout_account = holdout_account.copy()
df_holdout_tx = holdout_tx.copy()

def _build_timeseries_features(df_tx, windows=(7, 30, 90, 180)):
    df_work = df_tx[[LABEL_KEY, 'Tx_date', 'processing_volume', 'Tx_hours']].copy()
    df_work['Tx_date'] = pd.to_datetime(df_work['Tx_date'])
    df_work = df_work.sort_values([LABEL_KEY, 'Tx_date']).reset_index(drop=True)

    df_snapshots = (
        df_work[[LABEL_KEY, 'Tx_date']]
        .drop_duplicates([LABEL_KEY, 'Tx_date'])
        .copy()
    )

    for window in windows:
        df_window = (
            df_work
            .set_index('Tx_date')
            .groupby(LABEL_KEY)[['processing_volume', 'Tx_hours']]
            .rolling(f'{window}D', min_periods=1)
            .agg(['mean', 'min', 'max', 'std'])
            .reset_index()
        )

        df_window.columns = [
            LABEL_KEY, 'Tx_date',
            'avg_proc_vol', 'min_proc_vol', 'max_proc_vol', 'std_proc_vol',
            'avg_tx_hours', 'min_tx_hours', 'max_tx_hours', 'std_tx_hours'
        ]

        df_window[f'cv_proc_vol_{window}d'] = (
            df_window['std_proc_vol'] / df_window['avg_proc_vol'].replace(0, pd.NA)
        )
        df_window[f'cv_tx_hours_{window}d'] = (
            df_window['std_tx_hours'] / df_window['avg_tx_hours'].replace(0, pd.NA)
        )

        rename_map = {
            'avg_proc_vol': f'avg_proc_vol_{window}d',
            'min_proc_vol': f'min_proc_vol_{window}d',
            'max_proc_vol': f'max_proc_vol_{window}d',
            'avg_tx_hours': f'avg_tx_hours_{window}d',
            'min_tx_hours': f'min_tx_hours_{window}d',
            'max_tx_hours': f'max_tx_hours_{window}d',
        }

        keep_cols = [LABEL_KEY, 'Tx_date'] + list(rename_map.keys()) + [
            f'cv_proc_vol_{window}d',
            f'cv_tx_hours_{window}d',
        ]

        df_window = df_window[keep_cols].rename(columns=rename_map)
        df_snapshots = df_snapshots.merge(df_window, on=[LABEL_KEY, 'Tx_date'], how='left')

    # Momentum features: percent change vs exact lag date (current vs N days ago).
    df_daily = (
        df_work[[LABEL_KEY, 'Tx_date', 'processing_volume', 'Tx_hours']]
        .drop_duplicates([LABEL_KEY, 'Tx_date'])
        .rename(columns={
            'processing_volume': 'curr_processing_volume',
            'Tx_hours': 'curr_tx_hours',
        })
    )
    df_snapshots = df_snapshots.merge(df_daily, on=[LABEL_KEY, 'Tx_date'], how='left')

    for lag_days in windows:
        lag_df = df_daily.rename(columns={
            'curr_processing_volume': f'proc_vol_{lag_days}d_ago',
            'curr_tx_hours': f'tx_hours_{lag_days}d_ago',
        }).copy()
        lag_df['Tx_date'] = lag_df['Tx_date'] + pd.Timedelta(days=lag_days)

        df_snapshots = df_snapshots.merge(
            lag_df,
            on=[LABEL_KEY, 'Tx_date'],
            how='left',
        )

        df_snapshots[f'pct_change_proc_vol_vs_{lag_days}d_ago'] = (
            (df_snapshots['curr_processing_volume'] - df_snapshots[f'proc_vol_{lag_days}d_ago'])
            / df_snapshots[f'proc_vol_{lag_days}d_ago'].replace(0, pd.NA)
        )
        df_snapshots[f'pct_change_tx_hours_vs_{lag_days}d_ago'] = (
            (df_snapshots['curr_tx_hours'] - df_snapshots[f'tx_hours_{lag_days}d_ago'])
            / df_snapshots[f'tx_hours_{lag_days}d_ago'].replace(0, pd.NA)
        )

    df_snapshots['snapshot_day_of_week'] = df_snapshots['Tx_date'].dt.dayofweek
    df_snapshots['snapshot_day_of_year'] = df_snapshots['Tx_date'].dt.dayofyear
    df_snapshots['snapshot_month'] = df_snapshots['Tx_date'].dt.month
    df_snapshots['snapshot_quarter'] = df_snapshots['Tx_date'].dt.quarter

    return df_snapshots.sort_values([LABEL_KEY, 'Tx_date']).reset_index(drop=True)


df_train_features = _build_timeseries_features(df_train_tx)
df_holdoutfeatures = _build_timeseries_features(df_holdout_tx)

# Remove source file index artifacts before merge (avoids Unnamed: 0_x / Unnamed: 0_y).
df_train_account_clean = df_train_account.drop(columns=[c for c in df_train_account.columns if c.startswith('Unnamed:')], errors='ignore')
df_train_label_clean = df_train_label.drop(columns=[c for c in df_train_label.columns if c.startswith('Unnamed:')], errors='ignore')
df_holdout_account_clean = df_holdout_account.drop(columns=[c for c in df_holdout_account.columns if c.startswith('Unnamed:')], errors='ignore')

# Merge requested datasets
# train = account + features + labels
# holdout = account + features
df_train_merged = (
    df_train_features
    .merge(df_train_account_clean, on=LABEL_KEY, how='left')
    .merge(df_train_label_clean, on=LABEL_KEY, how='left')
)
df_holdout_merged = df_holdoutfeatures.merge(df_holdout_account_clean, on=LABEL_KEY, how='left')

print('df_train_features shape:', df_train_features.shape)
print('df_holdoutfeatures shape:', df_holdoutfeatures.shape)
print('df_train_merged shape:', df_train_merged.shape)
print('df_holdout_merged shape:', df_holdout_merged.shape)
print('Any Unnamed columns (train)?', any(c.startswith('Unnamed:') for c in df_train_merged.columns))
print('Any Unnamed columns (holdout)?', any(c.startswith('Unnamed:') for c in df_holdout_merged.columns))

df_train_features shape: (3510679, 56)
df_holdoutfeatures shape: (1471016, 56)
df_train_merged shape: (3510679, 60)
df_holdout_merged shape: (1471016, 59)
Any Unnamed columns (train)? False
Any Unnamed columns (holdout)? False


In [10]:
df_holdout_merged.head()

,Restaurant_ID,Tx_date,avg_proc_vol_7d,min_proc_vol_7d,max_proc_vol_7d,avg_tx_hours_7d,min_tx_hours_7d,max_tx_hours_7d,cv_proc_vol_7d,cv_tx_hours_7d,...,tx_hours_180d_ago,pct_change_proc_vol_vs_180d_ago,pct_change_tx_hours_vs_180d_ago,snapshot_day_of_week,snapshot_day_of_year,snapshot_month,snapshot_quarter,Ownership_type,Restaurant_catagory,Market_segment
0,00122837-df5b-496a-9eab-58da8b13f3b2,2021-07-02,11621.650000,11621.65,11621.65,9.00,9.0,9.0,NaN,NaN,...,NaN,NaN,NaN,4,183,7,3,LLC,FSR - Casual Dining,SMB
1,00122837-df5b-496a-9eab-58da8b13f3b2,2021-07-03,15284.535000,11621.65,18947.42,10.00,9.0,11.0,0.338911,0.141421,...,NaN,NaN,NaN,5,184,7,3,LLC,FSR - Casual Dining,SMB
2,00122837-df5b-496a-9eab-58da8b13f3b2,2021-07-04,10223.023333,100.00,18947.42,7.00,1.0,11.0,0.929396,0.755929,...,NaN,NaN,NaN,6,185,7,3,LLC,FSR - Casual Dining,SMB
3,00122837-df5b-496a-9eab-58da8b13f3b2,2021-07-05,7667.267500,0.00,18947.42,5.25,0.0,11.0,1.211684,1.0591,...,NaN,NaN,NaN,0,186,7,3,LLC,FSR - Casual Dining,SMB
4,00122837-df5b-496a-9eab-58da8b13f3b2,2021-07-06,6133.814000,0.00,18947.42,4.20,0.0,11.0,1.425841,1.275533,...,NaN,NaN,NaN,1,187,7,3,LLC,FSR - Casual Dining,SMB


### Account preprocessing

Normalize ownership strings and add `Restaurant_cat` (first three characters of category).


In [11]:
def normalize_ownership_type(x):
    if x == 'SoleProprietorship':
        return 'Sole Proprietorship'
    if x == 'PrivateCorporation':
        return 'Private Corporation'
    if x == 'PublicCorporation':
        return 'Public Corporation'
    if x == 'SECRegulatedCorporation':
        return 'SEC Regulated Corporation'
    if x == 'NonProfit':
        return 'Non Profit'
    if x == 'Corporate':
        return 'Corporation'
    return x

# Fix: Use the DataFrame's own columns for filtering, not another DataFrame's columns
df_train_merged = df_train_merged.loc[:, ~df_train_merged.columns.str.contains('^Unnamed')].copy()
df_holdout_merged = df_holdout_merged.loc[:, ~df_holdout_merged.columns.str.contains('^Unnamed')].copy()

df_train_merged['Ownership_type'] = df_train_merged['Ownership_type'].apply(normalize_ownership_type)
df_holdout_merged['Ownership_type'] = df_holdout_merged['Ownership_type'].apply(normalize_ownership_type)

df_train_merged['Restaurant_cat'] = df_train_merged['Restaurant_catagory'].str[:3]
df_holdout_merged['Restaurant_cat'] = df_holdout_merged['Restaurant_catagory'].str[:3]

In [12]:
df = df_train_merged
df_unique_n = df.nunique(axis=0).to_frame().rename(columns={0:'unique_values'})
df_unique_n.reset_index(inplace=True)
df_unique_n = df_unique_n.rename(columns = {'index':'feature_name'})

In [ ]:
# Low-cardinality distributions (sampled for speed — full df has ~3.5M rows per restaurant-day)
plot_df = df.sample(n=min(50_000, len(df)), random_state=42) if len(df) > 50_000 else df
low_card_cols = list(df_unique_n[df_unique_n['unique_values'] <= 30]['feature_name'])
for col in low_card_cols:
    fig, ax = plt.subplots(figsize=(15, 3))
    sns.countplot(data=plot_df, x=col, ax=ax)
    ax.tick_params(axis='x', rotation=45)
    ax.set_title(col)
    plt.tight_layout()
